# Day 4: Deep Dive into QUBO 
Welcome to Day 4! Today we will learn what a **QUBO** is, how to convert our portfolio optimization problem into one, and how to understand **penalty terms**.

## 1. What is a QUBO?
QUBO stands for **Quadratic Unconstrained Binary Optimization**. 
- **Binary**: Every choice we make is a yes/no (1 or 0). Here, $x_i = 1$ means we buy stock $i$, and $x_i = 0$ means we skip it.
- **Unconstrained**: We do not have hard limits (like "choose exactly $k$ stocks") in the main formula. Instead, we use **penalties** to punish the system if it breaks our rules.
- **Quadratic**: The equation goes up to squares ($x_i \times x_j$). We do not have cubes or higher powers. This is perfect for matrices!

The standard form of a QUBO is to **Minimize**: 
$$f(x) = \sum_{i} \sum_{j} Q_{ij} x_i x_j$$
where $Q$ is our special matrix (the Q-matrix) that holds all the returns, risks, and penalties.

---
## 2. Our Portfolio Problem Formulation
We want to do three things:
1. **Maximize Return**: represented by expected returns $\mu_i$
2. **Minimize Risk**: represented by the covariance matrix $\sigma_{ij}$
3. **Constraint**: Pick exactly $k$ stocks.

### The Math Formula
To build the objective function to minimize:

**Objective = - (Return) + $q$ * (Risk) + $\lambda$ * (Penalty)**

- $\mathbf{\mu_i}$: Expected return
- $\mathbf{\sigma_{ij}}$: Covariance (risk)
- $\mathbf{q}$: Risk weight (how scared are we of risk?)
- $\mathbf{\lambda}$: Penalty multiplier (how strongly do we enforce picking exactly $k$ stocks?)

$$ \text{Minimize } f(x) = -\sum_{i} \mu_i x_i + q \sum_{i,j} \sigma_{ij} x_i x_j + \lambda \left( \sum_{i} x_i - k \right)^2 $$

---
## 3. The Penalty Term Explained
Why the formula $\left( \sum x_i - k \right)^2$? 

- If we select exactly $k$ stocks, then $\sum x_i = k$, so the penalty becomes $(k - k)^2 = 0$. No punishment! We obeyed the rule.
- If we select $k+1$ or $k-1$ stocks, it becomes $(1)^2 = 1$ or $(-1)^2 = 1$. The formula adds $\lambda \times 1$ to the cost. If $\lambda$ is large (like 10 or 100), this makes the "cost" of that portfolio huge, so the minimizer avoids it.

When we expand this penalty mathematically, its pieces nicely fall into the diagonal ($Q_{ii}$) and off-diagonal ($Q_{ij}$) entries of our $Q$ matrix.

In [ ]:
import numpy as np
import pandas as pd
from itertools import product

np.set_printoptions(suppress=True, precision=4)

## 4. Building the Q Matrix in Code
Instead of importing a function from another file, let's write the `build_Q_matrix` logic right here so we can see exactly how the math turns into code.

In [2]:
def build_Q_matrix(returns, cov, penalty, k, risk_weight=1.0):
    n = len(returns)
    Q = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i == j:
                # DIAGONAL: -Return + Risk + Penalty Diagonal
                Q[i][i] = (
                    -returns[i]                    # Maximize return (so we make it negative to minimize)
                    + risk_weight * cov[i][i]      # Risk: variance of the asset
                    + penalty * (1 - 2 * k)        # Penalty: from expanding the squared penalty term
                )
            else:
                # OFF-DIAGONAL: Risk + Penalty Cross-terms
                Q[i][j] = (
                    risk_weight * cov[i][j]        # Risk: covariance between assets i and j
                    + penalty                      # Penalty: from expanding the squared penalty term
                )

    # Make sure Q is symmetrical
    Q = (Q + Q.T) / 2
    return Q

## 5. The Brute Force Solver
Since we're just learning and only have a few stocks right now, we can write a simple brute-force solver that checks EVERY possible binary portfolio ($2^n$ combinations) and finds the absolute best one.

In [3]:
def brute_force_solve(Q, n):
    best_x = None
    best_obj = float("inf")  # Start with an infinitely bad score

    # Generate all binary strings of length n (e.g. 000, 001, 010...)
    for bits in product([0, 1], repeat=n):
        x = np.array(bits, dtype=int)
        
        # Calculate the objective function: f(x) = x^T * Q * x
        obj = float(x @ Q @ x)

        if obj < best_obj:
            best_obj = obj
            best_x = x.copy()

    return best_x, best_obj

def decode_bitstring(x, tickers):
    # Simply maps the [0, 1, 1...] array to actual stock names
    return [tickers[i] for i in range(len(x)) if x[i] == 1]

## 6. Putting it into practice
Let's load the data from your verified GitHub repository. Fetching it straight from GitHub guarantees this notebook will work universally on any Google Colab or local CPU instance!

In [7]:
# Hardcoded portfolio data for demonstration (to work in any environment)
# In production, load from cached files

tickers = ['ICICIBANK', 'KOTAKBANK', 'AXISBANK', 'HDFCBANK', 'SBIN', 'INDUSINDBK']

mu = np.array([ 0.06574643,  0.0016126,   0.05589102, -0.00123439,  0.16337352, -0.35367911])

sigma = np.array([
    [0.03706484, 0.01711054, 0.02571261, 0.02044438, 0.02369213, 0.01307968],
    [0.01711054, 0.0565081,  0.02112788, 0.01885764, 0.02126277, 0.01829955],
    [0.02571261, 0.02112788, 0.05814683, 0.02445718, 0.03337956, 0.0280938 ],
    [0.02044438, 0.01885764, 0.02445718, 0.03741472, 0.02076008, 0.01852383],
    [0.02369213, 0.02126277, 0.03337956, 0.02076008, 0.0646312,  0.03324001],
    [0.01307968, 0.01829955, 0.0280938,  0.01852383, 0.03324001, 0.15364106]
])

print(f"Using hardcoded data for {len(tickers)} stocks: {tickers}")
print("Expected Returns (mu):", np.round(mu, 4))

Using hardcoded data for 6 stocks: ['ICICIBANK', 'KOTAKBANK', 'AXISBANK', 'HDFCBANK', 'SBIN', 'INDUSINDBK']
Expected Returns (mu): [ 0.0657  0.0016  0.0559 -0.0012  0.1634 -0.3537]


### Run the Optimization
We use `k=2` and `lambda=10`.

In [8]:
k = 2             # We want exactly 2 stocks
penalty = 10.0    # Hard penalty to prevent picking 1 or 3 stocks
risk_weight = 0.5 

Q = build_Q_matrix(mu, sigma, penalty=penalty, k=k, risk_weight=risk_weight)
n_assets = len(tickers)

best_x, best_obj = brute_force_solve(Q, n_assets)

print(f"Best binary string found: {best_x}")
print(f"Objective cost: {best_obj:.4f}\n")
best_portfolio = decode_bitstring(best_x, tickers)
print(f"The optimal {k}-stock portfolio is: {best_portfolio}")

Best binary string found: [1 0 0 0 1 0]
Objective cost: -40.1546

The optimal 2-stock portfolio is: ['ICICIBANK', 'SBIN']
